[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeiraDavidson/math_module2/blob/main/ch1_sequences.ipynb)

# Chapter 1 — Companion Notebook
## Patterns That Go Somewhere
*What does a list of numbers want to become?*

Three live experiments on the central idea of the chapter — that a never-ending list can be *sneaking up on* a single number. You will play the $\varepsilon$–$N$ game against a shrinking tolerance, watch a race between a base and an exponent decide a $1^\infty$ limit, and climb a cobweb staircase to a fixed point you cannot write in closed form.

> **How to use this notebook.** Run every cell from the top (Shift+Enter). Each widget below is live — drag the sliders and watch the picture respond. Requires `ipywidgets` and `matplotlib`.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from ipywidgets import (interact, interactive, fixed, Dropdown, Checkbox,
                        Button, Output, VBox, HBox,
                        FloatSlider, IntSlider, FloatLogSlider)
from IPython.display import HTML, display, Markdown

# Book 2 palette — matches the printed figures in figures/png/
BLUE="#1f4e79"; RED="#c0392b"; GREEN="#2e8b57"; ORANGE="#e08e0b"
PURPLE="#6c3483"; TEAL="#1b9e9e"; GRAY="#7f8c8d"; DARK="#2c3e50"
SHADE="#9fc5e8"; SHADE2="#f4b6ad"; SHADEG="#a9dfbf"; LIGHT="#d6dbdf"

plt.rcParams.update({
    "figure.figsize": (7.2, 4.5), "figure.dpi": 96,
    "font.family": "serif", "mathtext.fontset": "cm", "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": LIGHT, "grid.linewidth": 0.7,
    "lines.linewidth": 2.2, "lines.solid_capstyle": "round",
})

def axes(ax=None, title=None):
    '''A clean axes with a light grid; returns the axes.'''
    if ax is None:
        _, ax = plt.subplots()
    if title:
        ax.set_title(title, color=DARK)
    return ax


## 1. Name your tolerance — the $\varepsilon$–$N$ game

Here is the definition turned into a game. A skeptic names a tolerance $\varepsilon>0$ and bets your terms cannot get within $\varepsilon$ of the limit $L$ *and stay there*. You answer with a **cutoff** $N$: a seat number past which **every** term is inside the window $L\pm\varepsilon$. If you can win for any $\varepsilon$ the skeptic dares to name, then $a_n\to L$.

Pick a sequence and drag $\varepsilon$. The shaded band is $L\pm\varepsilon$; terms still *outside* it are grey, terms from the cutoff $N$ onward are green. Watch what happens as you shrink $\varepsilon$ toward zero: the band pinches, and the cutoff $N$ marches to the right. That trade-off *is* the limit.

In [2]:
# The convergent sequences on offer, each with its true limit L.
_SEQS = {
    'a_n = 1/n  ->  0':            (lambda n: 1.0 / n,            0.0),
    'a_n = n/(n+1)  ->  1':        (lambda n: n / (n + 1.0),      1.0),
    'a_n = (-1)^n / n  ->  0':     (lambda n: (-1.0) ** n / n,    0.0),
    'a_n = (1/2)^n  ->  0':        (lambda n: 0.5 ** n,           0.0),
}
_NTERMS = 40

def _epsilon_N(seq='a_n = 1/n  ->  0', eps=0.2):
    rule, L = _SEQS[seq]
    n = np.arange(1, _NTERMS + 1)
    a = rule(n)
    inside = np.abs(a - L) < eps          # term is within the window
    # Cutoff N = smallest index past which ALL later terms stay inside.
    # Scan from the right: the last term that is still OUTSIDE fixes N.
    outside_idx = np.where(~inside)[0]
    N = int(n[outside_idx[-1]]) if outside_idx.size else 0
    past = n > N                          # the green tail

    fig, ax = plt.subplots()
    axes(ax, f'{seq}    (tolerance \u03b5 = {eps:.4g})')
    ax.axhline(L, color=DARK, lw=1.3, zorder=1)
    ax.axhspan(L - eps, L + eps, color=SHADE, alpha=0.45, zorder=0,
               label=f'window  L \u00b1 \u03b5')
    ax.scatter(n[~past], a[~past], s=34, color=GRAY, zorder=3,
               label='before cutoff')
    ax.scatter(n[past], a[past], s=34, color=GREEN, zorder=3,
               label='from N onward (all inside)')
    ax.axvline(N + 0.5, color=RED, lw=1.6, ls='--', zorder=2)
    ax.annotate(f'N = {N}', xy=(N + 0.5, L), xytext=(N + 2.5, L + 1.4 * eps),
                color=RED, fontsize=13,
                arrowprops=dict(arrowstyle='->', color=RED))
    ax.set_xlabel('n  (seat number)'); ax.set_ylabel('a_n')
    ax.set_xlim(0, _NTERMS + 1)
    pad = max(eps * 1.6, 0.05)
    lo = min(a.min(), L - eps) - pad
    hi = max(a.max(), L + eps) + pad
    ax.set_ylim(lo, hi)
    ax.legend(loc='upper right', framealpha=0.9, fontsize=9)
    plt.show()

interact(_epsilon_N,
         seq=Dropdown(options=list(_SEQS), value='a_n = 1/n  ->  0',
                      description='sequence'),
         eps=FloatLogSlider(value=0.2, base=10, min=-3, max=0, step=0.05,
                            description='\u03b5', readout_format='.4g'));

interactive(children=(Dropdown(description='sequence', options=('a_n = 1/n  ->  0', 'a_n = n/(n+1)  ->  1', 'a…

## 2. A race for $1^\infty$ — where does $\left(1+\tfrac{b}{n}\right)^{pn}$ land?

The form $1^\infty$ is indeterminate because the answer hinges on a *race*: how fast the base $1+\tfrac{b}{n}$ sinks toward $1$ against how fast the exponent $pn$ climbs. The gauge that settles it: look at $(\text{exponent})\times(\text{base}-1)$. Here that is $pn\cdot\tfrac{b}{n}=b\,p$, a constant — so the sequence lands on

$$\left(1+\frac{b}{n}\right)^{pn}\longrightarrow e^{\,b\,p}.$$

Drag the base-rate $b$ and the power $p$. The blue dots are the sequence; the red line is the predicted limit $e^{bp}$. Push the product $bp$ toward $0$ and the limit slides to $1$; crank it up and the limit shoots toward $\infty$. Set $b=p=1$ to recover the birth of $e$ itself.

In [3]:
def _race(b=1.0, p=1.0):
    n = np.arange(1, 121)
    a = (1.0 + b / n) ** (p * n)
    k = b * p                       # the gauge:  (exponent)x(base-1)
    limit = np.exp(k)

    fig, ax = plt.subplots()
    axes(ax, f'(1 + {b:.2g}/n)^({p:.2g}\u00b7n)   \u2192   e^(b\u00b7p) = e^{{{k:.3g}}}')
    ax.scatter(n, a, s=20, color=BLUE, zorder=3, label='the sequence')
    ax.axhline(limit, color=RED, lw=2.0, ls='--', zorder=2,
               label=f'limit  e^(bp) = {limit:.5g}')
    ax.axhline(1.0, color=GRAY, lw=1.0, ls=':', zorder=1, label='y = 1')
    ax.set_xlabel('n'); ax.set_ylabel('a_n')
    lo = min(1.0, a.min(), limit); hi = max(a.max(), limit)
    span = hi - lo or 1.0
    ax.set_ylim(lo - 0.08 * span, hi + 0.12 * span)
    ax.legend(loc='best', framealpha=0.9, fontsize=9)
    # A one-line read-out of which way the race tipped.
    if abs(k) < 0.05:
        verdict = 'b\u00b7p \u2248 0: the base wins, limit swings toward 1.'
    elif k > 0:
        verdict = f'b\u00b7p = {k:.3g} > 0: balanced, limit = e^{k:.3g} = {limit:.4g}.'
    else:
        verdict = f'b\u00b7p = {k:.3g} < 0: limit = e^{k:.3g} = {limit:.4g} (below 1).'
    ax.text(0.5, -0.22, verdict, transform=ax.transAxes, ha='center',
            color=DARK, fontsize=10)
    plt.show()

interact(_race,
         b=FloatSlider(value=1.0, min=-2.0, max=3.0, step=0.1,
                       description='base-rate b'),
         p=FloatSlider(value=1.0, min=0.2, max=4.0, step=0.1,
                       description='power p'));

interactive(children=(FloatSlider(value=1.0, description='base-rate b', max=3.0, min=-2.0), FloatSlider(value=…

## 3. Climbing to a fixed point — the cobweb for $a_{n+1}=\sqrt{2+a_n}$

Some sequences have no closed form, yet still settle on an exact limit. Take $a_1=\sqrt{2}$ and $a_{n+1}=\sqrt{2+a_n}$. It climbs: $\sqrt2,\ \sqrt{2+\sqrt2},\ \ldots$, bounded above by $2$. If it converges to $L$, then $L$ must satisfy its own rule, $L=\sqrt{2+L}$, which gives $L=2$ — a **fixed point**, a value the process leaves unchanged.

The **cobweb** makes this visible. The blue curve is $y=\sqrt{2+x}$; the grey line is $y=x$. From a term on the diagonal, go *up* to the curve (that computes the next term), then *across* to the diagonal (that copies it back onto the input axis), and repeat. Drag the number of steps and watch the staircase spiral into the corner at $(2,2)$ where the two graphs cross. The dots on the right are the same terms, marching to $2$.

In [ ]:
def _g(x):
    return np.sqrt(2.0 + x)

def _cobweb(steps=6):
    # Build the orbit a_1 = sqrt(2), a_{n+1} = sqrt(2 + a_n).
    a = [np.sqrt(2.0)]
    for _ in range(steps):
        a.append(_g(a[-1]))
    a = np.array(a)

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(10.4, 4.6))

    # --- left: the cobweb diagram ---
    axes(axL, 'cobweb:  y = \u221a(2 + x)  vs  y = x')
    xs = np.linspace(0, 2.6, 400)
    axL.plot(xs, _g(xs), color=BLUE, label='y = \u221a(2 + x)')
    axL.plot(xs, xs, color=GRAY, lw=1.4, ls='--', label='y = x')
    axL.plot(2, 2, 'o', color=RED, ms=8, zorder=5)
    axL.annotate('fixed point (2, 2)', xy=(2, 2), xytext=(0.9, 2.35),
                 color=RED, fontsize=10,
                 arrowprops=dict(arrowstyle='->', color=RED))
    # staircase: (x, x) -> (x, g(x)) -> (g(x), g(x)) -> ...
    x = a[0]
    axL.plot([x, x], [0, _g(x)], color=GREEN, lw=1.4, zorder=4)
    for i in range(steps):
        y = _g(x)
        axL.plot([x, y], [y, y], color=GREEN, lw=1.4, zorder=4)   # across to y=x
        axL.plot([y, y], [y, _g(y)], color=GREEN, lw=1.4, zorder=4)  # up to curve
        x = y
    axL.set_xlabel('x = a_n'); axL.set_ylabel('a_{n+1}')
    axL.set_xlim(0, 2.6); axL.set_ylim(0, 2.6)
    axL.legend(loc='lower right', framealpha=0.9, fontsize=9)

    # --- right: the terms approaching 2 ---
    axes(axR, 'the terms a_n \u2192 2')
    idx = np.arange(1, len(a) + 1)
    axR.axhline(2.0, color=RED, lw=1.6, ls='--', label='limit L = 2')
    axR.plot(idx, a, '-o', color=BLUE, ms=6, zorder=3)
    for xi, yi in zip(idx, a):
        axR.annotate(f'{yi:.4f}', xy=(xi, yi), xytext=(0, 7),
                     textcoords='offset points', ha='center',
                     fontsize=8, color=DARK)
    axR.set_xlabel('n'); axR.set_ylabel('a_n')
    axR.set_xlim(0.5, len(a) + 0.5); axR.set_ylim(1.3, 2.08)
    axR.legend(loc='lower right', framealpha=0.9, fontsize=9)
    fig.tight_layout()
    plt.show()

interact(_cobweb,
         steps=IntSlider(value=6, min=1, max=12, step=1,
                         description='iterations'));